⭐⭐런타임 재연결 시 실행 코드⭐⭐
Google Drive 연결 + 경로 정의 + 저장된 split 불러오기 코드

EGD-CXR처럼 다시 다운받기 번거로운 데이터는 Drive쪽에 저장. (런타임 끊겨도 유지)

/content/drive/MyDrive ➡ 내 드라이브

/content/drive/MyDrive/X-ray ➡ 연구 프로젝트 폴더

In [ ]:
from google.colab import drive
drive.mount("/content/drive")  # 드라이브 마운트 경로

⭐⭐기본 import + 경로 + cohort 불러오기⭐⭐

In [ ]:
from pathlib import Path
import pandas as pd

EGD_DATA_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/egd-cxr/"
    "physionet.org/files/egd-cxr/1.0.0"
)

SPLIT_ROOT = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/splits"
)

cohort = pd.read_csv(
    SPLIT_ROOT / "egd_cxr_cohort_with_split_seed42.csv"
)

patient_split = pd.read_csv(
    SPLIT_ROOT / "patient_split_seed42_70_15_15.csv"
)

print("EGD 데이터 존재: ", EGD_DATA_ROOT.exists())
print("cohort: ", cohort.shape)
print("patient split: ", patient_split.shape)

⭐⭐실제 JPG 경로 다시 붙이기⭐⭐

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from PIL import Image

# 프로젝트 루트
PROJECT_ROOT = Path("/content/drive/MyDrive/X-ray")

# 다운로드한 MIMIC-CXR JPG 폴더
IMAGE_ROOT = (
    PROJECT_ROOT
    / "data/mimic-cxr-jpg-egdcxr/images"
)

# heatmap 정보까지 저장된 cohort CSV
COHORT_PATH = (
    PROJECT_ROOT
    / "gaze_calibration_research"
    / "splits"
    / "egd_cxr_cohort_with_split_and_heatmap_seed42.csv"
)

# cohort 다시 불러오기
cohort_with_heatmap = pd.read_csv(COHORT_PATH)

print("PROJECT_ROOT 존재:", PROJECT_ROOT.exists())
print("IMAGE_ROOT 존재:", IMAGE_ROOT.exists())
print("COHORT_PATH 존재:", COHORT_PATH.exists())
print("cohort shape:", cohort_with_heatmap.shape)

In [ ]:
from pathlib import Path

BASE_IMAGE_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images/physionet.org"
)

cohort_with_heatmap["image_path"] = (
    cohort_with_heatmap["path"]
    .astype(str)
    .apply(lambda p: str(BASE_IMAGE_ROOT / p.replace(".dcm", ".jpg")))
)

print(
    "이미지 경로 누락:",
    cohort_with_heatmap["image_path"].isna().sum()
)

print(
    "실제 존재하지 않는 이미지:",
    (~cohort_with_heatmap["image_path"].apply(lambda p: Path(p).exists())).sum()
)

print(
    "heatmap 경로 누락:",
    cohort_with_heatmap["heatmap_path"].isna().sum()
)

⭐⭐label 다시 만들기⭐⭐

In [ ]:
label_map = {
    "Normal": 0,
    "CHF": 1,
    "Pneumonia": 2
}

cohort_with_heatmap["label"] = (
    cohort_with_heatmap["target"].map(label_map)
)

print(
    cohort_with_heatmap[["target", "label"]]
    .drop_duplicates()
    .sort_values("label")
)

⭐⭐Dataset 클래스 + transform 다시 정의⭐⭐

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset

image_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class EGDCXRDataset(Dataset):
    def __init__(self, df, image_transform=None):
        self.df = df.reset_index(drop=True)
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"]).convert("RGB")

        if self.image_transform is not None:
            img = self.image_transform(img)

        heatmap = np.load(
            row["heatmap_path"]
        ).astype(np.float32)

        heatmap = torch.from_numpy(
            heatmap
        ).unsqueeze(0)

        label = torch.tensor(
            int(row["label"]),
            dtype=torch.long
        )

        return {
            "image": img,
            "heatmap": heatmap,
            "label": label,
            "dicom_id": row["dicom_id"]
        }

⭐⭐train / val / test Dataset + DataLoader 다시 만들기⭐⭐

In [ ]:
from torch.utils.data import DataLoader

train_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "train"
].copy()

val_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "val"
].copy()

test_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "test"
].copy()

train_dataset = EGDCXRDataset(
    train_df,
    image_transform=image_transform
)

val_dataset = EGDCXRDataset(
    val_df,
    image_transform=image_transform
)

test_dataset = EGDCXRDataset(
    test_df,
    image_transform=image_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

⭐⭐DenseNet-121 Baseline 모델 다시 정의⭐⭐

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("device:", device)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

model = densenet121(
    weights=DenseNet121_Weights.IMAGENET1K_V1
)

# 3-class classification으로 변경
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, 3)

model = model.to(device)

print(model.classifier)

⭐⭐Loss와 Optimizer 정의⭐⭐

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("criterion / optimizer 정의 완료")

⭐⭐GPU에서 한 배치 forward 테스트⭐⭐

In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():
    logits = model(images)

loss = criterion(logits, labels)

print("image device:", images.device)
print("logits shape:", logits.shape)
print("labels shape:", labels.shape)
print("loss:", loss.item())

⭐⭐checkpoint/history 저장 경로 정의⭐⭐

In [ ]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/checkpoints/baseline_densenet121"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pt"
HISTORY_PATH = CHECKPOINT_DIR / "history.pt"

print(CHECKPOINT_DIR)

⭐⭐Baseline 학습 함수 다시 정의⭐⭐

In [ ]:
from tqdm.auto import tqdm
import torch

def train_baseline(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10,
    start_epoch=1,
    best_val_loss=float("inf"),
    history=None
):
    # 처음 학습할 때
    if history is None:
        history = {
            "epoch": [],
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": []
        }

    for epoch in range(start_epoch, epochs + 1):

        print(f"\n===== Epoch {epoch}/{epochs} =====")

        # =====================
        # Train
        # =====================
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_bar = tqdm(
            train_loader,
            desc=f"Train {epoch}/{epochs}"
        )

        for batch in train_bar:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            batch_size = images.size(0)

            train_loss += loss.item() * batch_size

            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += batch_size

            current_loss = train_loss / train_total
            current_acc = train_correct / train_total

            train_bar.set_postfix(
                loss=f"{current_loss:.4f}",
                acc=f"{current_acc:.4f}"
            )

        train_loss /= train_total
        train_acc = train_correct / train_total

        # =====================
        # Validation
        # =====================
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        val_bar = tqdm(
            val_loader,
            desc=f"Val {epoch}/{epochs}"
        )

        with torch.no_grad():
            for batch in val_bar:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                batch_size = images.size(0)

                val_loss += loss.item() * batch_size

                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += batch_size

                current_val_loss = val_loss / val_total
                current_val_acc = val_correct / val_total

                val_bar.set_postfix(
                    loss=f"{current_val_loss:.4f}",
                    acc=f"{current_val_acc:.4f}"
                )

        val_loss /= val_total
        val_acc = val_correct / val_total

        # =====================
        # History 저장
        # =====================
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # history 자체도 Drive 저장
        torch.save(history, HISTORY_PATH)

        # =====================
        # Epoch 결과 출력
        # =====================
        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_acc:.4f}"
        )

        # =====================
        # Best model 저장
        # =====================
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "val_loss": val_loss,
                    "val_acc": val_acc,
                    "history": history
                },
                BEST_MODEL_PATH
            )

            print("→ Best model 저장")

        # =====================
        # 마지막 완료 epoch 저장
        # =====================
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "history": history
            },
            LAST_CHECKPOINT_PATH
        )

        print("→ Last checkpoint 저장")
        print("→ History 저장")

    print("\n학습 완료")
    print("Best val loss:", best_val_loss)

    return model, history



---



프로젝트 폴더 생성

```
X-ray/
├── EGD_CXR_Gaze_Calibration.ipynb
└── data/
    └── egd-cxr/
```



In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/X-ray") # 전체 연구 프로젝트 위치
DATA_ROOT = PROJECT_ROOT / "data"   # 데이터 전용 폴더
EGD_ROOT = DATA_ROOT / "egd-cxr"    # EGD-CXR 전용 폴더

EGD_ROOT.mkdir(parents=True, exist_ok=True)   # 실제 폴더 생성

print("프로젝트 폴더: ", PROJECT_ROOT)
print("데이터 폴더: ", DATA_ROOT)
print("EGD-CXR 폴더: ", EGD_ROOT)
print("EGD-CXR 폴더 존재: ", EGD_ROOT.exists())

PhysioNet에서 EGD-CXR 다운로드


```
egd-cxr/
└── physionet.org/
    └── files/
        └── egd-cxr/
            └── 1.0.0/
```



In [ ]:
import os

os.chdir(EGD_ROOT)
print("현재 작업 폴더: ", os.getcwd())

In [ ]:
!wget -r -N -c -np --user mijin --ask-password https://physionet.org/files/egd-cxr/1.0.0/

실제 대이터 폴더 확인



```
최상위 파일/폴더:
- LICENSE.txt
- SHA256SUMS.txt
- audio_segmentation_transcripts
- bounding_boxes.csv
- eye_gaze.csv
- fixations.csv
- inclusion_exclusion_criteria_outputs
- index.html
- master_sheet.csv
- table_descriptions.pdf
```



In [ ]:
from pathlib import Path

EGD_DATA_ROOT = EGD_ROOT / "physionet.org" / "files" / "egd-cxr" / "1.0.0"

print("EGD-CXR 실제 데이터 폴더:")
print(EGD_DATA_ROOT)

print("\n폴더 존재 여부:")
print(EGD_DATA_ROOT.exists())

print("\n최상위 파일/폴더:")
for item in sorted(EGD_DATA_ROOT.iterdir()):
    print("-", item.name)

다운로드 무결성 검사



```
======================================================================
SHA256 검증 완료
======================================================================
전체 파일: 7,593
정상 파일: 7,590
손상/불일치: 0
누락 파일: 3
총 소요시간: 23.0분

❌ 누락 파일:
- audio_segmentation_transcripts/0a1d6c66-b214a6e8-09501125-827f5b5a-3fa1bcc5/.DS_Store
- audio_segmentation_transcripts/0a90c3bb-3cf21ec9-835a1537-faf370a1-21053fd6/.DS_Store
- audio_segmentation_transcripts/feefc68a-0a26d0a0-65e4a2a0-b1efc19d-0abcd7d1/.DS_Store
➡ 연구에는 필요없는 파일이라 다운로드 검증 완료로 처리.
```



In [ ]:
from pathlib import Path
import hashlib
import time

# SHA256 목록 파일
checksum_file = EGD_DATA_ROOT / "SHA256SUMS.txt"

# 1. SHA256SUMS.txt 읽기
entries = []

with open(checksum_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        expected_hash, relative_path = line.split(maxsplit=1)

        # sha256sum 형식에서 파일명 앞의 *가 있을 경우 제거
        relative_path = relative_path.lstrip("*")

        entries.append((expected_hash, relative_path))

total = len(entries)

print(f"검사할 파일 수: {total:,}개")
print("SHA256 검증 시작...\n")

start_time = time.time()

ok_count = 0
failed = []
missing = []

# 2. 파일 하나씩 검사
for i, (expected_hash, relative_path) in enumerate(entries, start=1):

    file_path = EGD_DATA_ROOT / relative_path

    # 파일이 아예 없는 경우
    if not file_path.exists():
        missing.append(relative_path)

    else:
        sha256 = hashlib.sha256()

        with open(file_path, "rb") as f:
            while True:
                chunk = f.read(1024 * 1024)   # 1 MB씩 읽기

                if not chunk:
                    break

                sha256.update(chunk)

        actual_hash = sha256.hexdigest()

        if actual_hash == expected_hash:
            ok_count += 1
        else:
            failed.append(relative_path)

    # 10개마다 진행 상황 출력
    if i % 10 == 0 or i == total:

        elapsed = time.time() - start_time
        speed = i / elapsed

        remaining = total - i

        if speed > 0:
            eta = remaining / speed
        else:
            eta = 0

        print(
            f"[{i:,}/{total:,}] "
            f"{i/total*100:5.1f}% | "
            f"OK: {ok_count:,} | "
            f"경과: {elapsed/60:.1f}분 | "
            f"예상 남은 시간: {eta/60:.1f}분 | "
            f"현재: {relative_path}"
        )

# 3. 최종 결과
elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("SHA256 검증 완료")
print("=" * 70)

print(f"전체 파일: {total:,}")
print(f"정상 파일: {ok_count:,}")
print(f"손상/불일치: {len(failed):,}")
print(f"누락 파일: {len(missing):,}")
print(f"총 소요시간: {elapsed/60:.1f}분")

if len(failed) == 0 and len(missing) == 0:
    print("\n✅ 모든 파일이 정상입니다.")
else:
    if failed:
        print("\n❌ SHA256 불일치 파일:")
        for p in failed:
            print("-", p)

    if missing:
        print("\n❌ 누락 파일:")
        for p in missing:
            print("-", p)

EGD-CXR 데이터 확인

실제 파일들의 크기, 행 수. 컬럼명 확인

➡ 공통 연결 키: DICOM ID

➡ eye_gaze.csv: 프레임/샘플 단위의 더 촘촘한 원시 gaze 기록.
➡ fixations.csv: eye_gaze.csv 중 fixation 단위로 정리된 데이터.
  > FPOGX, FPOGY, FPOGD, X_ORIGINAL, Y_ORIGINAL 같은 fixation 위치와 지속시간 관련 값. (같은 DICOM에서 행 수가 훨씬 적을 것.)

따라서, 첫 실험에서는 fixations.csv부터 쓰는 것이 적절.



---

**bounding_boxes.csv**: cardiac silhouette, left clavicle, left lower lung zone 같은 **해부학적 영역 박스**가 포함되어 있음.



In [ ]:
#모든 파일X, 처음 몇 행만 읽어서 구조만 파악

import pandas as pd

files = {
    "master_sheet": EGD_DATA_ROOT / "master_sheet.csv",
    "fixations": EGD_DATA_ROOT / "fixations.csv",
    "eye_gaze": EGD_DATA_ROOT / "eye_gaze.csv",
    "bounding_boxes": EGD_DATA_ROOT / "bounding_boxes.csv",
}

for name, path in files.items():
    df_sample = pd.read_csv(path, nrows=5)

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print("파일 크기:", round(path.stat().st_size / (1024**2), 2), "MB")
    print("컬럼 수:", len(df_sample.columns))

    print("\n컬럼명:")
    print(df_sample.columns.tolist())

    print("\n처음 5행:")
    display(df_sample)

table_descriptions.pdf 내용 확인.
🐾크롬에서 PDF 텍스트 바로 뽑기.

In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils

In [ ]:
import subprocess

pdf_path = EGD_DATA_ROOT / "table_descriptions.pdf"
txt_path = EGD_DATA_ROOT / "table_descriptions.txt"

subprocess.run([
    "pdftotext",
    str(pdf_path),
    str(txt_path)
])

with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

print(text[:12000])

연구 타깃 결정

> pneumonia: ED 퇴원 ICD-9 진단코드로 판별한 임상적 폐렴 진단.

> pneumonia_chx: MIMIC-CXR 리포트에 CheXpert labeler를 적용해 얻은 영상 리포트 기반 라벨.

Normal / CHF / Pneumonia 3-class가 메인 실험으로 실제 가능한지 검증하는 cohort feasibility 단계

In [ ]:
import pandas as pd

# ============================================================
# 1. 데이터 불러오기
# ============================================================

master = pd.read_csv(EGD_DATA_ROOT / "master_sheet.csv")
fix = pd.read_csv(EGD_DATA_ROOT / "fixations.csv")

class_cols = ["Normal", "CHF", "pneumonia"]

print("=" * 70)
print("1. 기본 데이터 규모")
print("=" * 70)

print("전체 이미지 행 수:", len(master))
print("고유 DICOM 수:", master["dicom_id"].nunique())
print("고유 study 수:", master["study_id"].nunique())
print("고유 patient 수:", master["patient_id"].nunique())


# ============================================================
# 2. 세 임상 클래스의 양성 수 확인
# ============================================================

print("\n" + "=" * 70)
print("2. Normal / CHF / Pneumonia 클래스 분포")
print("=" * 70)

for col in class_cols:
    positive = (master[col] == 1).sum()
    print(f"{col:10s}: {positive:,}개")


# ============================================================
# 3. 클래스 중복 확인
# ============================================================

# 1인 경우에만 해당 클래스 양성으로 처리
class_binary = pd.DataFrame({
    col: (master[col] == 1).astype(int)
    for col in class_cols
})

print("\n" + "=" * 70)
print("3. 클래스 조합")
print("=" * 70)

combination = (
    class_binary
    .value_counts()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

print(combination.to_string(index=False))


# 한 영상에 몇 개 클래스가 동시에 양성인지
class_binary["positive_count"] = class_binary[class_cols].sum(axis=1)

print("\n한 영상의 양성 클래스 개수:")
print(
    class_binary["positive_count"]
    .value_counts()
    .sort_index()
)

print(
    "\n2개 이상 클래스가 동시에 양성인 이미지:",
    (class_binary["positive_count"] >= 2).sum()
)


# ============================================================
# 4. Fixation 데이터가 실제로 연결되는지 확인
# ============================================================

print("\n" + "=" * 70)
print("4. Gaze / Fixation 연결 가능성")
print("=" * 70)

master_ids = set(master["dicom_id"].astype(str))
fix_ids = set(fix["DICOM_ID"].dropna().astype(str))

print("master_sheet DICOM 수:", len(master_ids))
print("fixation이 있는 DICOM 수:", len(fix_ids))
print("둘 다 존재하는 DICOM 수:", len(master_ids & fix_ids))


# ============================================================
# 5. 클래스별 gaze 사용 가능 이미지 수
# ============================================================

tmp = master.copy()

tmp["dicom_id"] = tmp["dicom_id"].astype(str)
tmp["has_fixation"] = tmp["dicom_id"].isin(fix_ids)

print("\n" + "=" * 70)
print("5. 클래스별 fixation 보유 이미지")
print("=" * 70)

for col in class_cols:

    total_positive = (tmp[col] == 1).sum()

    gaze_positive = (
        (tmp[col] == 1) &
        (tmp["has_fixation"])
    ).sum()

    print(
        f"{col:10s}: "
        f"전체 {total_positive:,}개 / "
        f"fixation 있음 {gaze_positive:,}개"
    )


# ============================================================
# 6. '순수한' 3-class 후보 cohort 확인
# ============================================================

# Normal / CHF / Pneumonia 중 정확히 하나만 1인 경우
tmp["class_positive_count"] = (
    tmp[class_cols]
    .eq(1)
    .sum(axis=1)
)

pure = tmp[
    (tmp["class_positive_count"] == 1) &
    (tmp["has_fixation"])
].copy()

def assign_class(row):
    if row["Normal"] == 1:
        return "Normal"
    elif row["CHF"] == 1:
        return "CHF"
    elif row["pneumonia"] == 1:
        return "Pneumonia"
    return None

pure["target"] = pure.apply(assign_class, axis=1)

print("\n" + "=" * 70)
print("6. 겹치지 않는 3-class 후보 cohort")
print("=" * 70)

print(pure["target"].value_counts())

print("\n3-class 후보 전체 이미지:", len(pure))
print("3-class 후보 고유 환자:", pure["patient_id"].nunique())


# ============================================================
# 7. 각 클래스의 환자 수
# ============================================================

print("\n" + "=" * 70)
print("7. 클래스별 고유 환자 수")
print("=" * 70)

patient_counts = (
    pure.groupby("target")["patient_id"]
    .nunique()
    .sort_index()
)

print(patient_counts)

환자 단위 label consistency 검사



---

같은 환자가 서로 다른 클래스에 걸쳐있지 않음.

➡ patient_id를 기준으로 Normal / CHF / Pneumonia를 stratification해서 나누면, 환자 leakage(테스트 데이터가 훈련 과정에서 섞여 들어가는 현상)를 막으면서 클래스 비율도 유지 가능.

In [ ]:
# ============================================================
# 다음 단계:
# 같은 환자가 여러 class에 등장하는지 확인
# ============================================================

patient_target = (
    pure.groupby("patient_id")["target"]
        .agg(
            image_count="count",
            target_count="nunique",
            targets=lambda x: sorted(x.unique())
        )
        .reset_index()
)

print("=" * 70)
print("1. 환자당 이미지 수")
print("=" * 70)

print(
    patient_target["image_count"]
    .value_counts()
    .sort_index()
)

print("\n여러 이미지를 가진 환자 수:",
      (patient_target["image_count"] > 1).sum())


print("\n" + "=" * 70)
print("2. 같은 환자가 여러 class에 등장하는지")
print("=" * 70)

multi_target = patient_target[
    patient_target["target_count"] > 1
].copy()

print("여러 class에 속하는 환자 수:", len(multi_target))

if len(multi_target) > 0:
    display(multi_target)
else:
    print("✅ 모든 환자의 class가 일관적입니다.")


print("\n" + "=" * 70)
print("3. 여러 이미지를 가진 환자 예시")
print("=" * 70)

display(
    patient_target[
        patient_target["image_count"] > 1
    ].head(20)
)

지금까지 확인한 데이터 기준 메인 실험



```
                       EGD-CXR
                          │
                1,083 CXR + Fixation
                          │
            ┌─────────────┼─────────────┐
            │             │             │
         Normal          CHF        Pneumonia
          360            363           360
            │             │             │
            └─────────────┼─────────────┘
                          │
                  Patient-level split
                          │
          ┌───────────────┼────────────────┐
          │               │                │
       Baseline      Original Gaze     Controls
                                      ├─ Shuffled
                                      ├─ Random
                                      └─ ROI/Anatomy
                          │
             ┌────────────┴────────────┐
             │                         │
       Discrimination              Reliability
       AUROC / F1 etc.        ECE / Brier / NLL
                                     │
                           High-confidence errors
                                     │
                            Risk-Coverage / AURC
```



연구에서 모든 실험군이 공통으로 사용할 고정 train/validation/test split을 생성

단순 분류 연구X, calibration 연구 진행 예정. 따라서 validation set이 매우 중요

(Calibration 연구: 인공신경망인 센서 등의 출력값과 신뢰도를 실제 값이나 확률에 정확히 일치시키는 기술.)


*   Train: 모델 가중치 학습
*   Validation: early stopping, hyperparameter 선택, temperature scaling fitting
*   Test: 마지막 성능 평가만.

⭐ temperautre scaling에 test set을 사용하면 leakage가 생기기 때문에 반드시 calidation을 별도로 보존해야한다.





1차 메인 split: **70/15/15**

patient-level split 생성 + Drive 저장

> 런타임이 끊겨도 Drive 마운트 + EGD_DATA_ROOT 설정 후 이 셀만 다시 돌려도 됨.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================
# 1. master_sheet 불러오기
# ============================================================

master = pd.read_csv(EGD_DATA_ROOT / "master_sheet.csv")

class_cols = ["Normal", "CHF", "pneumonia"]


# ============================================================
# 2. 3-class target 생성
# ============================================================

def assign_target(row):
    if row["Normal"] == 1:
        return "Normal"
    elif row["CHF"] == 1:
        return "CHF"
    elif row["pneumonia"] == 1:
        return "Pneumonia"
    return None


master["target"] = master.apply(assign_target, axis=1)

# 세 클래스 중 정확히 하나만 양성인지 재검증
master["positive_count"] = master[class_cols].eq(1).sum(axis=1)

cohort = master[
    (master["positive_count"] == 1) &
    (master["target"].notna())
].copy()

print("전체 cohort:", len(cohort))
print(cohort["target"].value_counts())


# ============================================================
# 3. 환자별 대표 target 생성
# ============================================================

patient_df = (
    cohort.groupby("patient_id")
          .agg(
              target=("target", "first"),
              target_count=("target", "nunique"),
              image_count=("dicom_id", "count")
          )
          .reset_index()
)

# 혹시라도 한 환자가 여러 target이면 즉시 중단
assert patient_df["target_count"].max() == 1, \
    "한 환자가 여러 target에 속합니다."

print("\n고유 환자 수:", len(patient_df))
print("\n환자 기준 클래스 분포:")
print(patient_df["target"].value_counts())


# ============================================================
# 4. Patient-level 70 / 15 / 15 split
# ============================================================

RANDOM_SEED = 42

train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=patient_df["target"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_patients["target"]
)


# split 이름 부여
train_patients = train_patients.copy()
val_patients = val_patients.copy()
test_patients = test_patients.copy()

train_patients["split"] = "train"
val_patients["split"] = "val"
test_patients["split"] = "test"

patient_split = pd.concat(
    [train_patients, val_patients, test_patients],
    ignore_index=True
)


# ============================================================
# 5. 이미지 단위 cohort에 split 연결
# ============================================================

cohort_split = cohort.merge(
    patient_split[["patient_id", "split"]],
    on="patient_id",
    how="left"
)

assert cohort_split["split"].isna().sum() == 0


# ============================================================
# 6. Leakage 검사
# ============================================================

train_ids = set(
    cohort_split.loc[
        cohort_split["split"] == "train",
        "patient_id"
    ]
)

val_ids = set(
    cohort_split.loc[
        cohort_split["split"] == "val",
        "patient_id"
    ]
)

test_ids = set(
    cohort_split.loc[
        cohort_split["split"] == "test",
        "patient_id"
    ]
)

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\n✅ Patient leakage 없음")


# ============================================================
# 7. Split 결과 확인
# ============================================================

print("\n" + "=" * 70)
print("환자 기준 split")
print("=" * 70)

print(
    pd.crosstab(
        patient_split["split"],
        patient_split["target"],
        margins=True
    )
)


print("\n" + "=" * 70)
print("이미지 기준 split")
print("=" * 70)

print(
    pd.crosstab(
        cohort_split["split"],
        cohort_split["target"],
        margins=True
    )
)


# ============================================================
# 8. Drive에 영구 저장
# ============================================================

SAVE_ROOT = Path(
    "/content/drive/MyDrive/X-ray/"
    "gaze_calibration_research/splits"
)

SAVE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

patient_split_path = (
    SAVE_ROOT /
    "patient_split_seed42_70_15_15.csv"
)

cohort_split_path = (
    SAVE_ROOT /
    "egd_cxr_cohort_with_split_seed42.csv"
)

patient_split.to_csv(
    patient_split_path,
    index=False
)

cohort_split.to_csv(
    cohort_split_path,
    index=False
)

print("\n✅ 저장 완료")
print("Patient split:")
print(patient_split_path)

print("\nImage cohort split:")
print(cohort_split_path)

EGD-CXE-JPG 다운로드

In [ ]:
import os
import subprocess
from pathlib import Path

# ============================================================
# 1. 저장 폴더
# ============================================================

DOWNLOAD_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"
)

DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. 다운로드 목록 파일 생성
# ============================================================

url_list_path = DOWNLOAD_ROOT.parent / "egd_cxr_1083_urls.txt"

base_url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"

urls = [
    base_url + p
    for p in cohort["jpg_relative_path"]
]

with open(url_list_path, "w") as f:
    for url in urls:
        f.write(url + "\n")

print("URL 수:", len(urls))
print("목록 파일:", url_list_path)

In [ ]:
!wget \
  -c \
  -N \
  -x \
  --cut-dirs=3 \
  --user mijin \
  --ask-password \
  --progress=bar:force:noscroll \
  -i "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/egd_cxr_1083_urls.txt" \
  -P "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"

In [ ]:
from pathlib import Path
import pandas as pd

DOWNLOAD_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"
)

SPLIT_ROOT = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/splits"
)

cohort = pd.read_csv(
    SPLIT_ROOT / "egd_cxr_cohort_with_split_seed42.csv"
)

# 실제 다운로드된 JPG 찾기
jpg_files = list(DOWNLOAD_ROOT.rglob("*.jpg"))

# 실제 JPG 파일명의 dicom_id
downloaded_ids = {p.stem for p in jpg_files}

# 연구에 필요한 dicom_id
expected_ids = set(cohort["dicom_id"].astype(str))

missing_ids = expected_ids - downloaded_ids
extra_ids = downloaded_ids - expected_ids

print("연구 대상 이미지:", len(expected_ids))
print("다운로드된 JPG:", len(jpg_files))
print("일치한 이미지:", len(expected_ids & downloaded_ids))
print("누락:", len(missing_ids))
print("추가 파일:", len(extra_ids))

if missing_ids:
    print("\n누락된 DICOM ID 예시:")
    print(list(missing_ids)[:10])

In [ ]:
from pathlib import Path
from PIL import Image

IMAGE_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"
)

jpg_files = list(IMAGE_ROOT.rglob("*.jpg"))

corrupt_files = []

for i, p in enumerate(jpg_files, 1):
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        corrupt_files.append((str(p), str(e)))

    if i % 100 == 0:
        print(f"{i}/{len(jpg_files)} 검사 완료")

print("\n전체 JPG:", len(jpg_files))
print("손상 파일:", len(corrupt_files))

if corrupt_files:
    print("\n손상 파일 예시:")
    for item in corrupt_files[:10]:
        print(item)

In [ ]:
from pathlib import Path
from PIL import Image

IMAGE_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"
)

jpg_files = list(IMAGE_ROOT.rglob("*.jpg"))

corrupt_files = []

for i, p in enumerate(jpg_files, 1):
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception as e:
        corrupt_files.append((str(p), str(e)))

    if i % 100 == 0:
        print(f"{i}/{len(jpg_files)} 검사 완료")

print("\n전체 JPG:", len(jpg_files))
print("손상 파일:", len(corrupt_files))

JPG해상도와 원본 DICOM 좌표계가 맞는지 확인.

➡ fixations.csv의 X_ORIGINAL, Y_ORIGINAL이 원본 DICOM 좌표 기준이라서, 우리가 받은 JPG와 크기가 다르면 시선점이 엉뚱한위치에 찍힐 수 있기 때문에 진행.

MIMIC-CXR 공식 metadata 파일을 다운.

In [ ]:
import subprocess
from getpass import getpass

PHYSIONET_USERNAME = "rlaalwls1220@hufs.ac.kr"

password = getpass("PhysioNet password: ")

METADATA_URL = (
    "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
    "mimic-cxr-2.0.0-metadata.csv.gz"
)

result = subprocess.run(
    [
        "wget",
        "-c",
        "--user", PHYSIONET_USERNAME,
        "--password", password,
        "-O", str(METADATA_PATH),
        METADATA_URL
    ]
)

print("종료 코드:", result.returncode)
print("파일 존재:", METADATA_PATH.exists())

if METADATA_PATH.exists():
    print("파일 크기:", METADATA_PATH.stat().st_size, "bytes")

In [ ]:
import pandas as pd

metadata = pd.read_csv(METADATA_PATH)

print("metadata shape:", metadata.shape)
print(metadata.columns.tolist())
print(metadata.head())

PIL의 img.size 순서 = (width, height)

metadata는 Columns = width, Rows = height

➡ 비교할땐 img.size == (Columns, Rows)

In [ ]:
from PIL import Image

# dicom_id -> metadata 행
meta_small = metadata[["dicom_id", "Rows", "Columns"]].copy()

# cohort 1083장만 추리기
cohort_meta = cohort[["dicom_id"]].merge(
    meta_small,
    on="dicom_id",
    how="left"
)

# 실제 파일 경로 매핑
path_by_id = {p.stem: p for p in jpg_files}

mismatch = []
missing_meta = []

for i, row in cohort_meta.iterrows():
    dicom_id = str(row["dicom_id"])

    if pd.isna(row["Rows"]) or pd.isna(row["Columns"]):
        missing_meta.append(dicom_id)
        continue

    p = path_by_id[dicom_id]

    with Image.open(p) as img:
        jpg_w, jpg_h = img.size

    expected_w = int(row["Columns"])
    expected_h = int(row["Rows"])

    if (jpg_w, jpg_h) != (expected_w, expected_h):
        mismatch.append({
            "dicom_id": dicom_id,
            "jpg_size": (jpg_w, jpg_h),
            "metadata_size": (expected_w, expected_h)
        })

print("전체 cohort:", len(cohort_meta))
print("metadata 누락:", len(missing_meta))
print("해상도 불일치:", len(mismatch))

if mismatch:
    print("\n불일치 예시:")
    for x in mismatch[:10]:
        print(x)

위의 결과:

전체 cohort: 1083

metadata 누락: 0

해상도 불일치: 0


➡ MIMIC-CXR JPG가 원본 DICOM과 동일한 픽셀 크기를 유지하고 있다는 것을 cohort에서 직접 확인.

**좌표계 1차 검증**


*   cohort 1,083장 전부 metadata 존재.
*   JPG 크기와 원본 DICOM의 Columns x Rows가 1,083장 모두 일치.

➡ 따라서 X_ORIGINAL, Y_ORIGINAL을 JPG 위에 올려볼 근거가 생김.



fixation 좌표 범위가 이미지 크기 안에 들어오는지 확인 (heatmap X)

In [ ]:
import pandas as pd
from pathlib import Path

FIXATION_PATH = EGD_DATA_ROOT / "fixations.csv"

fix = pd.read_csv(FIXATION_PATH)

print("fixations shape:", fix.shape)
print("컬럼 존재 확인:")
print("DICOM_ID" in fix.columns)
print("X_ORIGINAL" in fix.columns)
print("Y_ORIGINAL" in fix.columns)

cohort 1,083장에해당하는 fixation만 추리기.

In [ ]:
cohort_ids = set(cohort["dicom_id"].astype(str))

fix_cohort = fix[
    fix["DICOM_ID"].astype(str).isin(cohort_ids)
].copy()

print("전체 fixation:", len(fix))
print("cohort fixation:", len(fix_cohort))
print("포함된 DICOM 수:", fix_cohort["DICOM_ID"].nunique())

좌표 점위 검사

In [ ]:
fix_check = fix_cohort.merge(
    metadata[["dicom_id", "Rows", "Columns"]],
    left_on="DICOM_ID",
    right_on="dicom_id",
    how="left"
)

valid_xy = (
    fix_check["X_ORIGINAL"].notna() &
    fix_check["Y_ORIGINAL"].notna()
)

out_of_bounds = fix_check[
    valid_xy &
    (
        (fix_check["X_ORIGINAL"] < 0) |
        (fix_check["Y_ORIGINAL"] < 0) |
        (fix_check["X_ORIGINAL"] >= fix_check["Columns"]) |
        (fix_check["Y_ORIGINAL"] >= fix_check["Rows"])
    )
]

print("좌표 있는 fixation:", valid_xy.sum())
print("이미지 범위 밖 fixation:", len(out_of_bounds))

➡ 좌표 있는 fixation: 48747, 이미지 범위 밖 fixation: 481

즉, 약 99.0%의 fixation 좌표는 이미지 내부에 정상적으로 들어가고, 약 1%만 범위를 벗어남.

유효 좌표만 남기기

In [ ]:
valid_fix = fix_check[
    valid_xy &
    (fix_check["X_ORIGINAL"] >= 0) &
    (fix_check["Y_ORIGINAL"] >= 0) &
    (fix_check["X_ORIGINAL"] < fix_check["Columns"]) &
    (fix_check["Y_ORIGINAL"] < fix_check["Rows"])
].copy()

print("원래 좌표 있는 fixation:", valid_xy.sum())
print("유효 fixation:", len(valid_fix))
print("제외된 fixation:", valid_xy.sum() - len(valid_fix))
print("유효 DICOM 수:", valid_fix["DICOM_ID"].nunique())

시각적 정합성 확인

➡ 시선점이 실제 흉부 X-ray의 폐, 심장, 종격동 같은 해부학적 영역에 자연스럽게 분포하는지 확인.




*   점들이 이미지 바깥으로 대량으로 튀지 않음.
*   좌우 반전/상하 반전처럼 보이지 않음.
*   폐야, 종격동, 심장, 횡경막 주변에 자연스럽게 분포.
*   일부 점ㅁ이 주변부에도 있지만, radiologist scan pattern으로 충분히 나올 수 있는 수준.



In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# dicom_id -> 실제 JPG 경로
path_by_id = {p.stem: p for p in jpg_files}

# 각 클래스에서 fixation이 존재하는 첫 이미지 1장 선택
classes = ["Normal", "CHF", "pneumonia"]

for cls in classes:
    sample_id = cohort.loc[cohort[cls] == 1, "dicom_id"].iloc[0]

    img_path = path_by_id[sample_id]
    pts = valid_fix[valid_fix["DICOM_ID"] == sample_id]

    with Image.open(img_path) as img:
        plt.figure(figsize=(8, 8))
        plt.imshow(img, cmap="gray")

        plt.scatter(
            pts["X_ORIGINAL"],
            pts["Y_ORIGINAL"],
            s=20,
            alpha=0.6
        )

        plt.title(
            f"{cls} | DICOM: {sample_id}\n"
            f"Fixations: {len(pts)}"
        )
        plt.axis("off")
        plt.show()

fixation point ➡ heatmap



```
heatmap 원칙

*   입력 이미지 크기: 나중 학습용으로 통일된 크기 사용 → 우선 512 x 512 추천
*   fixation 좌표: 원본 좌표를 512 x 512로 선형 스케일링
*   가중치: fixation duration FPOGD 사용
*   퍼짐 정도: Gaussian blur 사용
*   최종값: 0~1 사이로 정규화
```



```
heatmap 생성 방식

*   heatmap이 비어 있지 않음
*   fixation이 많았던 위치에 강한 hotspot이 생김
*   overlay했을 때 흉부 구조 위에 자연스럽게 올라감
*   좌우/상하 반전 문제 없어 보임
```





전체 heatmap 생성해서 Drive에 저장

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import gaussian_filter

# =========================
# 설정
# =========================
TARGET_SIZE = 512
SIGMA = 7

HEATMAP_ROOT = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/heatmaps_512_sigma7"
)
HEATMAP_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# 함수
# =========================
def make_heatmap_for_dicom(dicom_id, valid_fix, metadata, target_size=512, sigma=7):
    pts = valid_fix[valid_fix["DICOM_ID"] == dicom_id].copy()

    meta_row = metadata[metadata["dicom_id"] == dicom_id].iloc[0]
    orig_h = int(meta_row["Rows"])
    orig_w = int(meta_row["Columns"])

    heat = np.zeros((target_size, target_size), dtype=np.float32)

    if "FPOGD" in pts.columns:
        weights = pts["FPOGD"].fillna(1.0).astype(float).values
    else:
        weights = np.ones(len(pts), dtype=np.float32)

    xs = pts["X_ORIGINAL"].values / orig_w * (target_size - 1)
    ys = pts["Y_ORIGINAL"].values / orig_h * (target_size - 1)

    for x, y, w in zip(xs, ys, weights):
        xi = int(round(x))
        yi = int(round(y))
        if 0 <= xi < target_size and 0 <= yi < target_size:
            heat[yi, xi] += float(w)

    heat = gaussian_filter(heat, sigma=sigma)

    if heat.max() > 0:
        heat = heat / heat.max()

    return heat.astype(np.float32)

# =========================
# 전체 생성
# =========================
records = []

for i, dicom_id in enumerate(cohort["dicom_id"].astype(str), 1):
    heat = make_heatmap_for_dicom(
        dicom_id=dicom_id,
        valid_fix=valid_fix,
        metadata=metadata,
        target_size=TARGET_SIZE,
        sigma=SIGMA
    )

    save_path = HEATMAP_ROOT / f"{dicom_id}.npy"
    np.save(save_path, heat)

    records.append({
        "dicom_id": dicom_id,
        "heatmap_path": str(save_path),
        "heatmap_max": float(heat.max()),
        "heatmap_sum": float(heat.sum())
    })

    if i % 100 == 0:
        print(f"{i}/{len(cohort)} 완료")

heatmap_df = pd.DataFrame(records)

print("\n전체 heatmap 생성 완료")
print("생성 수:", len(heatmap_df))
print("저장 폴더:", HEATMAP_ROOT)

생성 결과를 cohort에 붙여서 저장.

In [ ]:
COHORT_WITH_HEATMAP_PATH = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/splits/"
    "egd_cxr_cohort_with_split_and_heatmap_seed42.csv"
)

cohort_with_heatmap = cohort.merge(
    heatmap_df[["dicom_id", "heatmap_path", "heatmap_max", "heatmap_sum"]],
    on="dicom_id",
    how="left"
)

cohort_with_heatmap.to_csv(COHORT_WITH_HEATMAP_PATH, index=False)

print("저장 완료:", COHORT_WITH_HEATMAP_PATH)
print(cohort_with_heatmap.head())

생성이 잘 됐는지 확인

In [ ]:
print("cohort 수:", len(cohort_with_heatmap))
print("heatmap 경로 누락:", cohort_with_heatmap["heatmap_path"].isna().sum())
print("heatmap max가 0인 이미지 수:", (cohort_with_heatmap["heatmap_max"] <= 0).sum())
print("heatmap sum이 0인 이미지 수:", (cohort_with_heatmap["heatmap_sum"] <= 0).sum())

학습용 데이터셋 클래스 구성



```
A. Baseline
*   입력: X-ray 이미지
*   출력: 3-class label

B. Gaze-supervised model
*   입력: X-ray 이미지
*   출력: 3-class label
*   추가 supervision: heatmap

➡ 같은 포맷의heatmap으로 만들기.
```



PyTorch Dataset이 이미지·라벨·heatmap을 정확히 한 샘플 단위로 묶어 반환하는지 검증

In [ ]:
# cohort_with_heatmap에 실제 JPG 경로를 붙임.

from pathlib import Path

IMAGE_ROOT = Path(
    "/content/drive/MyDrive/X-ray/data/mimic-cxr-jpg-egdcxr/images"
)

# dicom_id -> 실제 jpg 경로
path_by_id = {
    p.stem: str(p)
    for p in IMAGE_ROOT.rglob("*.jpg")
}

cohort_with_heatmap["image_path"] = (
    cohort_with_heatmap["dicom_id"]
    .astype(str)
    .map(path_by_id)
)

print("전체:", len(cohort_with_heatmap))
print("image_path 누락:", cohort_with_heatmap["image_path"].isna().sum())
print("heatmap_path 누락:", cohort_with_heatmap["heatmap_path"].isna().sum())

In [ ]:
# 라벨을 숫자로 고정

label_map = {
    "Normal": 0,
    "CHF": 1,
    "Pneumonia": 2
}

cohort_with_heatmap["label"] = (
    cohort_with_heatmap["target"]
    .map(label_map)
)

print(
    cohort_with_heatmap[
        ["target", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
)

Datast 클래스 생성

In [ ]:
label_map = {
    "Normal": 0,
    "CHF": 1,
    "Pneumonia": 2
}

cohort_with_heatmap["label"] = (
    cohort_with_heatmap["target"]
    .map(label_map)
)

print(
    cohort_with_heatmap[
        ["target", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
)

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

image_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class EGDCXRDataset(Dataset):
    def __init__(self, df, image_transform=None):
        self.df = df.reset_index(drop=True)
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # X-ray
        img = Image.open(row["image_path"]).convert("RGB")

        if self.image_transform is not None:
            img = self.image_transform(img)

        # gaze heatmap
        heatmap = np.load(row["heatmap_path"]).astype(np.float32)
        heatmap = torch.from_numpy(heatmap).unsqueeze(0)

        # class label
        label = torch.tensor(
            int(row["label"]),
            dtype=torch.long
        )

        return {
            "image": img,
            "heatmap": heatmap,
            "label": label,
            "dicom_id": row["dicom_id"]
        }

print("EGDCXRDataset 정의 완료")

In [ ]:
train_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "train"
].copy()

train_dataset = EGDCXRDataset(
    train_df,
    image_transform=image_transform
)

sample = train_dataset[0]

print("image shape:", sample["image"].shape)
print("heatmap shape:", sample["heatmap"].shape)
print("label:", sample["label"])
print("dicom_id:", sample["dicom_id"])
print("heatmap min/max:",
      sample["heatmap"].min().item(),
      sample["heatmap"].max().item())

In [ ]:
from torch.utils.data import DataLoader

train_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "train"
].copy()

val_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "val"
].copy()

test_df = cohort_with_heatmap[
    cohort_with_heatmap["split"] == "test"
].copy()

train_dataset = EGDCXRDataset(train_df, image_transform=image_transform)
val_dataset = EGDCXRDataset(val_df, image_transform=image_transform)
test_dataset = EGDCXRDataset(test_df, image_transform=image_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

실제 배치 확인

In [ ]:
batch = next(iter(train_loader))

print("image batch:", batch["image"].shape)
print("heatmap batch:", batch["heatmap"].shape)
print("label batch:", batch["label"].shape)
print("labels:", batch["label"])

Baseline DenseNet-121 정의

➡ 아직 gaze loss는 넣지말고, 먼저 이미지 → 3-class 분류만 하는 기준 모델을 만듦

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)

model = densenet121(
    weights=DenseNet121_Weights.IMAGENET1K_V1
)

# 마지막 classifier를 3-class로 변경
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, 3)

model = model.to(device)

print(model.classifier)

loss와 optimizer 정의

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

학습 전체를 돌리기 전에, 배치 forward가 정상인지 먼저 확인.



```
logits shape: torch.Size([8, 3])
labels shape: torch.Size([8])
loss: 1.0799726247787476

➡ 통과
```



In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():
    logits = model(images)

print("logits shape:", logits.shape)
print("labels shape:", labels.shape)

loss = criterion(logits, labels)
print("loss:", loss.item())

Baseline DenseNet-121 학습 루프 + validation + best model 저장

학습 코드로 학습 함수부터 정의

In [ ]:
import copy
import torch

def train_baseline(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10
):
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):

        # =====================
        # Train
        # =====================
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch in train_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # =====================
        # Validation
        # =====================
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                val_loss += loss.item() * images.size(0)

                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_acc:.4f}"
        )

        # best validation loss 기준 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    # 가장 좋았던 모델 복원
    model.load_state_dict(best_state)

    print("\nBest val loss:", best_val_loss)

    return model

GPU 사용중인지 확인

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)

Baseline 학습 + Drive checkpoint 저장

In [ ]:
# checkpoint 폴더 생성

from pathlib import Path

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/X-ray/gaze_calibration_research/checkpoints/baseline_densenet121"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pt"

print("checkpoint 폴더:", CHECKPOINT_DIR)

기존 tain_baseline() 함수를 아래 코드로 변경

In [ ]:
import copy
import torch

def train_baseline(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=10,
    start_epoch=1,
    best_val_loss=float("inf")
):
    for epoch in range(start_epoch, epochs + 1):

        # =====================
        # Train
        # =====================
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch in train_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # =====================
        # Validation
        # =====================
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                val_loss += loss.item() * images.size(0)

                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_acc:.4f}"
        )

        # =====================
        # 마지막 상태 저장
        # =====================
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "train_loss": train_loss,
                "train_acc": train_acc
            },
            LAST_CHECKPOINT_PATH
        )

        # =====================
        # Best model 저장
        # =====================
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "val_loss": val_loss,
                    "val_acc": val_acc
                },
                BEST_MODEL_PATH
            )

            print("  → Best model 저장")

    print("\n학습 완료")
    print("Best val loss:", best_val_loss)

    return model

실제 학습

**train_baseline()**

➡ 매 epoch마다 Drive에 "last_checkpoint.pt"가 저장됨.

➡ validation loss가 개선될 때마다 "best_model.pt"도 갱신됨.



```
*   last_checkpoint.pt → 학습 이어하기용
*   best_model.pt → 최종 평가용
```



In [ ]:
baseline_model = train_baseline(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=10
)

학습 도중 런타임이 끊기면 아래 코드 실행하고, 그 이후에 위 학습코드 start_ephoch 변경해서 돌리기

In [ ]:
checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]

print("마지막 완료 epoch:", checkpoint["epoch"])
print("다음 시작 epoch:", start_epoch)